# Speech to text

This experiment evaluates the performance of speech-to-text systems for converting spoken user queries into text under practical deployment conditions. The objective is to analyse the trade-off between transcription accuracy and processing efficiency when transforming natural language speech input into textual queries that can be used by the semantic retrieval component of the object locator system. In this study, each spoken query is processed by a speech recognition model to generate a textual transcription. The generated transcription is then compared against the ground truth query to evaluate recognition performance. In addition to transcription quality, the time required for speech recognition is measured to assess the computational efficiency of each model during inference. The goal is to identify the most suitable speech-to-text solution for the voice interaction component of the proposed object locator system, balancing accurate transcription with practical real-time responsiveness.

The evaluation will be based on the following metrics:

- Word Error Rate (WER)
- Command Recognition Accuracy
- Speech Recognition Latency
- End-to-End Processing Time

Word Error Rate (WER) measures the difference between the predicted transcription and the ground truth text by calculating the proportion of substitutions, insertions, and deletions relative to the total number of words in the reference query. Lower WER values indicate higher transcription accuracy and more reliable speech recognition.

Command recognition accuracy measures the percentage of spoken queries where the transcription correctly preserves the key object keywords required for retrieval. Since the object locator system relies on correctly identifying the object name within the query, this metric evaluates whether the essential semantic meaning of the command is preserved even if minor transcription differences occur.

Speech recognition latency measures the time required for the speech recognition model to convert an audio query into text. Lower latency is important for maintaining responsive voice interaction within the application.

End-to-end processing time measures the total time taken from receiving the audio input to producing the final transcription result. This includes audio processing and model inference, and reflects the practical responsiveness experienced by the user during real-time interaction.

The models that will be evaluated include:
- Vosk
- Coqui STT
- Device Speech Recognition (evaluated but not tested due to complexity)

The models selected represent three common approaches to speech recognition deployment. Vosk Speech Recognition represents a lightweight offline speech recognition model that performs transcription locally without requiring internet connectivity. Coqui STT represents a neural speech recognition system that also runs locally but uses deep learning techniques to achieve higher transcription accuracy. Device speech recognition represents built-in speech recognition systems provided by mobile operating systems, which are optimized for real-time interaction and low-latency processing on the device. These models were selected to compare the trade-offs between local processing, cloud-based recognition, and device-optimized speech recognition.

Six audio files are manually recording, each containing common phrases a user might say when trying to locate missing items, such as “where is my wallet” or “locate my phone.” A corresponding ground truth CSV file was created, where each audio file is paired with its correct transcription.

The model processes each audio file and generates a predicted transcription. This predicted text is then compared against the ground truth text, and the difference between them is quantified using Word Error Rate (WER), which measures how accurately the model transcribes the spoken queries

## VOSK

Below evaluates the performance of the speech recognition system using the VOSK offline ASR model. It processes a batch of audio files, compares the transcriptions generated by the model against ground truth text, and computes WER, latency and end to end time

In [ ]:
import os
import time
import json
import wave
import pandas as pd
from vosk import Model, KaldiRecognizer
from jiwer import wer

# paths
AUDIO_FOLDER = "audio_queries"
MODEL_PATH = "models/vosk-model-small-en-us-0.15"
GROUND_TRUTH_FILE = "ground_truth.csv"

# load the ground truth file into the pandas dataframe
gt = pd.read_csv(GROUND_TRUTH_FILE)

# load Vosk model
print("Loading Vosk model...")
model = Model(MODEL_PATH)

results = []

# warm up run using the first audio file
warmup_audio = os.path.join(AUDIO_FOLDER, gt.iloc[0]["file"])
with wave.open(warmup_audio, "rb") as wf:
    rec = KaldiRecognizer(model, wf.getframerate())
    # read audio file in chunks and feed it to the recognizer
    while True:
        data = wf.readframes(4000)
        if len(data) == 0:
            break
        rec.AcceptWaveform(data)
    # finalize recognition and discard the warm up result
    _ = json.loads(rec.FinalResult())

# iterating through each row in the ground truth dataframe
for _, row in gt.iterrows():
    # extract file name and corresponding truth transcription
    file_name = row["file"]
    true_text = row["text"]
    # constructing path to the audio file
    audio_path = os.path.join(AUDIO_FOLDER, file_name)

    with wave.open(audio_path, "rb") as wf:
        # initializing recognizer with the same sample rate as the audio file
        rec = KaldiRecognizer(model, wf.getframerate())

        start_time = time.time()
        # read audio data in chunks and process it through the recognizer
        while True:
            data = wf.readframes(4000)
            if len(data) == 0:
                break
            rec.AcceptWaveform(data)
        
        end_time = time.time()
    # parsing the final recognition result in JSON format
    result = json.loads(rec.FinalResult())
    predicted_text = result.get("text", "")

    latency = end_time - start_time

    error = wer(true_text, predicted_text)
    # appending results
    results.append({
        "file": file_name,
        "ground_truth": true_text,
        "prediction": predicted_text,
        "WER": error,
        "latency_sec": latency
    })

df = pd.DataFrame(results)


# checking if the key command word is correctly recognized
def command_correct(gt, pred):
    # assumes the last word in the ground truth sentence is the main keyword
    # returns true if the keyword appears anywhere in the predicted text
    keyword = gt.split()[-1]
    return keyword in pred
# apply the function to every row in the dataframe
df["command_correct"] = df.apply(
    lambda x: command_correct(x["ground_truth"], x["prediction"]), axis=1
)

# compute metrics
avg_wer = df["WER"].mean()
command_accuracy = df["command_correct"].mean()
avg_latency = df["latency_sec"].mean()

# end-to-end time (same as latency for this STT experiment)
end_to_end_time = avg_latency

# store final results into dataframe
results_df = pd.DataFrame([{
    "Model": "Vosk",
    "Word Error Rate (WER)": avg_wer,
    "Command Recognition Accuracy": command_accuracy,
    "Speech Recognition Latency (sec)": avg_latency,
    "End-to-End Processing Time (sec)": end_to_end_time
}])

results_df

Loading Vosk model...


,Model,Word Error Rate (WER),Command Recognition Accuracy,Speech Recognition Latency (sec),End-to-End Processing Time (sec)
0,Vosk,0.314286,0.5,0.968552,0.968552


## Coqui STT

The performance of the Coqui STT model is as evaluated below. The pipeline is the same as for when evaluating the performance of the VOSK model

In [ ]:
import os
import time
import wave
import pandas as pd
import numpy as np
from jiwer import wer
from stt import Model as CoquiModel

# paths
AUDIO_FOLDER = "audio_queries"
GROUND_TRUTH_FILE = "ground_truth.csv"
# loading the model and the scorer
COQUI_MODEL_PATH = "models/coqui/model.tflite"
COQUI_SCORER_PATH = "models/coqui/huge-vocabulary.scorer"

# load the ground truth file into the pandas dataframe
gt = pd.read_csv(GROUND_TRUTH_FILE)

# load Coqui model
print("Loading Coqui STT model...")
model = CoquiModel(COQUI_MODEL_PATH)

# optional scorer if detected is enabled
if os.path.exists(COQUI_SCORER_PATH):
    model.enableExternalScorer(COQUI_SCORER_PATH)

results = []

# warm up run using the first audio file
warmup_audio = os.path.join(AUDIO_FOLDER, gt.iloc[0]["file"])
with wave.open(warmup_audio, "rb") as wf:
    # reading all the audio frames in the file 
    frames = wf.readframes(wf.getnframes())
    # convert raw audio bytes into a NumPy array of 16-bit integers since that is the format expected from the model
    audio = np.frombuffer(frames, dtype=np.int16)
    _ = model.stt(audio)

# Evaluation loop
for _, row in gt.iterrows():
    # extract file name and corresponding truth transcription
    file_name = row["file"]
    true_text = row["text"]
    # constructing path to the audio file
    audio_path = os.path.join(AUDIO_FOLDER, file_name)

    with wave.open(audio_path, "rb") as wf:
        start_time = time.time()

        frames = wf.readframes(wf.getnframes())
        audio = np.frombuffer(frames, dtype=np.int16)
        # perform speech-to-text transcription and convert output to lowercase
        predicted_text = model.stt(audio).lower()

        end_time = time.time()

    latency = end_time - start_time

    error = wer(true_text, predicted_text)
    # appending results
    results.append({
        "file": file_name,
        "ground_truth": true_text,
        "prediction": predicted_text,
        "WER": error,
        "latency_sec": latency
    })

df = pd.DataFrame(results)

# checking if the key command word is correctly recognized
def command_correct(gt, pred):
    # assumes the last word in the ground truth sentence is the main keyword
    # returns true if the keyword appears anywhere in the predicted text
    keyword = gt.split()[-1]
    return keyword in pred
# apply the function to every row in the dataframe
df["command_correct"] = df.apply(
    lambda x: command_correct(x["ground_truth"], x["prediction"]), axis=1
)

# compute metrics
avg_wer = df["WER"].mean()
command_accuracy = df["command_correct"].mean()
avg_latency = df["latency_sec"].mean()
# end-to-end time (same as latency for this STT experiment)
end_to_end_time = avg_latency

# Append to results_df
new_row = pd.DataFrame([{
    "Model": "Coqui STT",
    "Word Error Rate (WER)": avg_wer,
    "Command Recognition Accuracy": command_accuracy,
    "Speech Recognition Latency (sec)": avg_latency,
    "End-to-End Processing Time (sec)": end_to_end_time
}])

try:
    results_df = pd.concat([results_df, new_row], ignore_index=True)
except NameError:
    results_df = new_row

results_df

Loading Coqui STT model...


,Model,Word Error Rate (WER),Command Recognition Accuracy,Speech Recognition Latency (sec),End-to-End Processing Time (sec)
0,Vosk,0.314286,0.5,0.968552,0.968552
1,Coqui STT,1.055556,0.0,3.045019,3.045019


The experimental evaluation compares two local speech recognition models to analyse the trade-off between lightweight and neural speech recognition approaches for command-based queries. The results show that Vosk achieves substantially lower Word Error Rate (0.314) compared to Coqui STT (1.056), indicating that Vosk provides more reliable transcription for the short command phrases used in the object locator system. In addition, Vosk demonstrates significantly lower speech recognition latency (approximately 0.97 seconds) compared to Coqui STT (approximately 3.05 seconds). This suggests that lightweight speech recognition models optimized for command-style speech can provide faster response times while maintaining reasonable recognition accuracy.

In contrast, Coqui STT exhibits both higher transcription errors and higher processing latency. This is likely because the evaluated Coqui model is designed for general speech transcription and performs less effectively when applied to short command-style inputs with limited contextual information. The results therefore indicate that lightweight speech recognition systems may be better suited for command-based interaction scenarios.

Although the experimental comparison evaluates local speech recognition models, practical mobile applications commonly rely on the device’s native speech recognition system provided by the operating system. Device speech recognition systems are typically optimized for real-time interaction, microphone input processing, and hardware acceleration. These systems are also trained on large-scale speech datasets and integrated directly with the device’s audio pipeline, enabling faster and more reliable speech recognition in real-world usage. Therefore, while the experimental results highlight the advantages of lightweight speech recognition models such as Vosk, the final implementation of the object locator system utilizes the device’s native speech recognition functionality to achieve improved responsiveness, integration with the mobile platform, and a more seamless user experience.